# Bài 5: Time Travel & Lịch sử phiên bản

## Mục tiêu
- Xem lịch sử thay đổi của bảng bằng `DESCRIBE HISTORY`.
- Đọc bảng tại 1 version/thời điểm cũ bằng `VERSION AS OF` / `TIMESTAMP AS OF`.
- Khôi phục bảng về version cũ bằng `RESTORE TABLE`.
- Hiểu mối liên hệ giữa time travel và log/data retention (liên quan `VACUUM` — Bài 8).


## 5.1. `DESCRIBE HISTORY`

Mỗi commit trong `_delta_log` tương ứng 1 dòng trong `DESCRIBE HISTORY`, gồm: `version`, `timestamp`, `operation` (WRITE/UPDATE/DELETE/MERGE/OPTIMIZE/RESTORE...), `operationParameters`, `operationMetrics` (số dòng thêm/xoá/sửa), `readVersion`, `isolationLevel`...

```sql
DESCRIBE HISTORY db.tbl;            -- toan bo lich su
DESCRIBE HISTORY db.tbl LIMIT 5;    -- 5 commit gan nhat
```

## 5.2. Đọc dữ liệu tại version/thời điểm cũ

```sql
SELECT * FROM db.tbl VERSION AS OF 3;
SELECT * FROM db.tbl TIMESTAMP AS OF '2024-01-15 10:00:00';
```
Hoặc bằng DataFrameReader:
```python
spark.read.format("delta").option("versionAsOf", 3).table("db.tbl")
spark.read.format("delta").option("timestampAsOf", "2024-01-15").table("db.tbl")
```

Cơ chế: Delta chỉ cần replay `_delta_log` từ commit `0` đến commit `N` (version yêu cầu) để biết chính xác tập file Parquet nào thuộc về version đó — **không cần** file dữ liệu bị đổi, vì file cũ (`remove`) vẫn còn nằm vật lý trên storage cho tới khi bị `VACUUM` dọn.

## 5.3. `RESTORE TABLE`

```sql
RESTORE TABLE db.tbl TO VERSION AS OF 3;
RESTORE TABLE db.tbl TO TIMESTAMP AS OF '2024-01-15 10:00:00';
```

`RESTORE` **không xoá lịch sử** — nó tạo ra **một commit mới** đưa bảng về đúng trạng thái của version cũ (add lại các file đã bị remove, remove các file được add sau đó). Vì vậy sau khi restore, `DESCRIBE HISTORY` vẫn thấy đầy đủ các version trước đó **và thêm 1 dòng operation = RESTORE** — bản thân việc restore cũng time-travel được.

## 5.4. Giới hạn thực tế

- Time travel **phụ thuộc retention**: file dữ liệu cũ chỉ tồn tại tới khi `VACUUM` xoá (mặc định giữ 7 ngày); log JSON cũ được kiểm soát bởi `delta.logRetentionDuration` (mặc định 30 ngày). Sau khi `VACUUM`, các version cũ tham chiếu tới file đã xoá sẽ **không đọc lại được** dù log vẫn còn.
- Càng nhiều version giữ lại → càng nhiều file nhỏ tồn đọng cho tới khi VACUUM → cần cân bằng giữa nhu cầu audit/rollback và chi phí lưu trữ + hiệu năng liệt kê file.


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai05"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai05-time-travel")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 5.5. Ví dụ minh hoạ

In [ ]:
spark.sql("DROP TABLE IF EXISTS bai05.scores")
spark.createDataFrame([(1,"An",80),(2,"Binh",70)], ["id","name","score"]).write.format("delta").saveAsTable("bai05.scores")
# version 0

spark.sql("UPDATE bai05.scores SET score = 90 WHERE id = 1")
# version 1

spark.createDataFrame([(3,"Chi",95)], ["id","name","score"]).write.format("delta").mode("append").saveAsTable("bai05.scores")
# version 2

spark.sql("DELETE FROM bai05.scores WHERE id = 2")
# version 3

spark.sql("SELECT * FROM bai05.scores ORDER BY id").show()


In [ ]:
spark.sql("DESCRIBE HISTORY bai05.scores").select("version","timestamp","operation","operationMetrics").show(truncate=False)


In [ ]:
print("== Version 0 (ngay sau khi tao bang) ==")
spark.sql("SELECT * FROM bai05.scores VERSION AS OF 0 ORDER BY id").show()

print("== Version 2 (truoc khi xoa id=2) ==")
spark.sql("SELECT * FROM bai05.scores VERSION AS OF 2 ORDER BY id").show()


In [ ]:
# Khoi phuc bang ve version 0 - tao them 1 commit RESTORE moi, khong mat lich su
spark.sql("RESTORE TABLE bai05.scores TO VERSION AS OF 0")
spark.sql("SELECT * FROM bai05.scores ORDER BY id").show()
spark.sql("DESCRIBE HISTORY bai05.scores").select("version","operation").show()


## 5.6. Thực hành

**Bài 1** — Tạo bảng `bai05.wallet (user_id INT, balance DOUBLE)`, sau đó thực hiện 3 thao tác thay đổi khác nhau (ví dụ: append, update, delete) để có ít nhất 4 version (0-3).

**Bài 2** — In `DESCRIBE HISTORY` đầy đủ, xác định version nào tương ứng operation nào.

**Bài 3** — Đọc dữ liệu bảng tại version `1` bằng cú pháp SQL `VERSION AS OF`, và tại version mới nhất bằng `DataFrameReader.option("versionAsOf", ...)`.

**Bài 4** — Dùng `RESTORE TABLE` để đưa bảng về version `1`. Kiểm tra `DESCRIBE HISTORY` sau khi restore — version mới nhất có `operation = 'RESTORE'` không? Số lượng dòng lịch sử tăng hay giảm?

**Bài 5 (tư duy)** — Giả sử bảng có `delta.deletedFileRetentionDuration = 'interval 1 hours'` và đã bị `VACUUM` chạy sau khoảng thời gian đó. Nếu bạn cố `SELECT ... VERSION AS OF 0` (version đã quá cũ so với retention), điều gì xảy ra? Vì sao?


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

_Viết câu trả lời của bạn ở đây._

---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1 & 2
spark.sql("DROP TABLE IF EXISTS bai05.wallet")
spark.createDataFrame([(1,100.0),(2,200.0)], ["user_id","balance"]).write.format("delta").saveAsTable("bai05.wallet")  # v0
spark.createDataFrame([(3,50.0)], ["user_id","balance"]).write.format("delta").mode("append").saveAsTable("bai05.wallet")  # v1
spark.sql("UPDATE bai05.wallet SET balance = balance + 10 WHERE user_id = 1")  # v2
spark.sql("DELETE FROM bai05.wallet WHERE user_id = 2")  # v3

spark.sql("DESCRIBE HISTORY bai05.wallet").select("version","operation","operationMetrics").show(truncate=False)


In [ ]:
# Dap an Bai 3
spark.sql("SELECT * FROM bai05.wallet VERSION AS OF 1 ORDER BY user_id").show()
spark.read.format("delta").option("versionAsOf", 3).table("bai05.wallet").orderBy("user_id").show()


In [ ]:
# Dap an Bai 4
spark.sql("RESTORE TABLE bai05.wallet TO VERSION AS OF 1")
spark.sql("SELECT * FROM bai05.wallet ORDER BY user_id").show()
spark.sql("DESCRIBE HISTORY bai05.wallet").select("version","operation").show()
# -> them 1 dong moi voi operation = RESTORE; lich su TANG them 1 dong, khong mat gi ca


**Đáp án Bài 5**: Sẽ ném lỗi (kiểu `DELTA_TIME_TRAVEL_INVALID_BEGIN_VALUE` / file-not-found). Vì `VACUUM` đã **xoá vật lý** các file Parquet không còn được tham chiếu bởi version hiện tại quá hạn retention. Version `0` trong `_delta_log` (JSON) có thể vẫn còn tồn tại (log retention thường dài hơn data retention), nhưng khi Delta cố đọc file Parquet mà version 0 trỏ tới, file đó đã bị xoá thật trên storage → lỗi "file not found". Đây là lý do phải cân nhắc kỹ `retentionDurationCheck` trước khi hạ thấp retention hoặc chạy `VACUUM` với retention ngắn.
